In [1]:
import re
import warnings
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

from imblearn.over_sampling import RandomOverSampler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import pandas as pd

DATA_PATH = "../../data/gerrit/couchbase_java_client/processed/gerrit_modeling_intermediate_expansion.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df["security_label_binary"].value_counts())

Dataset loaded successfully!
Shape: (1774, 18)
security_label_binary
0    1382
1     392
Name: count, dtype: int64


In [3]:
df = df.dropna(
    subset=["code_diff", "security_label_binary", "review_number"]
).copy()

df["code_diff"] = df["code_diff"].astype(str)
df["security_label_binary"] = df["security_label_binary"].astype(int)
df["review_number"] = df["review_number"].astype(str)

X_text = df["code_diff"].reset_index(drop=True)
y = df["security_label_binary"].reset_index(drop=True)
groups = df["review_number"].reset_index(drop=True)

print("Prepared dataset:", len(df))
print("Unique review groups:", groups.nunique())

print("\nTarget distribution:")
print(y.value_counts())

print("\nSecurity-related percentage:", round(y.mean() * 100, 2), "%")

Prepared dataset: 1774
Unique review groups: 808

Target distribution:
security_label_binary
0    1382
1     392
Name: count, dtype: int64

Security-related percentage: 22.1 %


In [4]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

def evaluate_model_features(X, y, groups, experiment_name):
    fold_results = []
    oof_predictions = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y), dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y, groups), start=1
    ):

        X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_test = X.iloc[test_idx] if hasattr(X, "iloc") else X[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Oversampling is applied ONLY to the training fold.
        ros = RandomOverSampler(random_state=RANDOM_STATE)

        X_train_resampled, y_train_resampled = ros.fit_resample(
            X_train, y_train
        )

        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

        model.fit(X_train_resampled, y_train_resampled)

        predictions = model.predict(X_test)
        scores = model.predict_proba(X_test)[:, 1]

        oof_predictions[test_idx] = predictions
        oof_scores[test_idx] = scores

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "pr_auc": average_precision_score(y_test, scores)
        })

    fold_df = pd.DataFrame(fold_results)

    mean_metrics = fold_df.drop(columns="fold").mean()

    pooled_metrics = {
        "accuracy": accuracy_score(y, oof_predictions),
        "balanced_accuracy": balanced_accuracy_score(y, oof_predictions),
        "precision": precision_score(y, oof_predictions, zero_division=0),
        "recall": recall_score(y, oof_predictions, zero_division=0),
        "f1": f1_score(y, oof_predictions, zero_division=0),
        "pr_auc": average_precision_score(y, oof_scores)
    }

    print(f"\n{experiment_name}")
    print("\nMean 5-fold metrics:")
    display(mean_metrics.to_frame("Mean Score"))

    print("\nPooled out-of-fold metrics:")
    display(pd.Series(pooled_metrics, name="Pooled OOF Score").to_frame())

    return {
        "experiment": experiment_name,
        "fold_results": fold_df,
        "mean_metrics": mean_metrics,
        "pooled_metrics": pooled_metrics,
        "oof_predictions": oof_predictions,
        "oof_scores": oof_scores
    }

print("Cross-validation and evaluation function ready.")

Cross-validation and evaluation function ready.


In [5]:
def evaluate_text_model(
    X_text,
    y,
    groups,
    experiment_name,
    extra_features=None,
    use_word2vec=False
):
    from scipy.sparse import hstack, csr_matrix
    from sklearn.feature_extraction.text import TfidfVectorizer

    fold_results = []
    oof_predictions = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y), dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X_text, y, groups), start=1
    ):
        train_text = X_text.iloc[train_idx].fillna("")
        test_text = X_text.iloc[test_idx].fillna("")

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            stop_words="english"
        )

        X_train = vectorizer.fit_transform(train_text)
        X_test = vectorizer.transform(test_text)

        if extra_features is not None:
            X_train_extra = csr_matrix(
                extra_features.iloc[train_idx].to_numpy()
            )
            X_test_extra = csr_matrix(
                extra_features.iloc[test_idx].to_numpy()
            )

            X_train = hstack([X_train, X_train_extra]).tocsr()
            X_test = hstack([X_test, X_test_extra]).tocsr()

        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_train_resampled, y_train_resampled = ros.fit_resample(
            X_train, y_train
        )

        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

        model.fit(X_train_resampled, y_train_resampled)

        predictions = model.predict(X_test)
        scores = model.predict_proba(X_test)[:, 1]

        oof_predictions[test_idx] = predictions
        oof_scores[test_idx] = scores

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "pr_auc": average_precision_score(y_test, scores)
        })

        print(f"Completed fold {fold}/5")

    fold_df = pd.DataFrame(fold_results)

    mean_metrics = fold_df.drop(columns="fold").mean()

    pooled_metrics = {
        "accuracy": accuracy_score(y, oof_predictions),
        "balanced_accuracy": balanced_accuracy_score(y, oof_predictions),
        "precision": precision_score(y, oof_predictions, zero_division=0),
        "recall": recall_score(y, oof_predictions, zero_division=0),
        "f1": f1_score(y, oof_predictions, zero_division=0),
        "pr_auc": average_precision_score(y, oof_scores)
    }

    print(f"\n{experiment_name}")
    print("\nMean 5-fold metrics:")
    display(mean_metrics.to_frame("Mean Score"))

    print("\nPooled out-of-fold metrics:")
    display(pd.Series(pooled_metrics, name="Pooled OOF Score").to_frame())

    return {
        "experiment": experiment_name,
        "fold_results": fold_df,
        "mean_metrics": mean_metrics,
        "pooled_metrics": pooled_metrics,
        "oof_predictions": oof_predictions,
        "oof_scores": oof_scores
    }


print("Fold-specific evaluation function ready.")

Fold-specific evaluation function ready.


In [6]:
baseline_results = evaluate_text_model(
    X_text=X_text,
    y=y,
    groups=groups,
    experiment_name="Baseline: TF-IDF + Logistic Regression + Random Oversampling"
)

Completed fold 1/5
Completed fold 2/5
Completed fold 3/5
Completed fold 4/5
Completed fold 5/5

Baseline: TF-IDF + Logistic Regression + Random Oversampling

Mean 5-fold metrics:


,Mean Score
accuracy,0.885560
balanced_accuracy,0.866212
precision,0.705725
recall,0.831581
f1,0.762291
pr_auc,0.861571



Pooled out-of-fold metrics:


,Pooled OOF Score
accuracy,0.885569
balanced_accuracy,0.866250
precision,0.704104
recall,0.831633
f1,0.762573
pr_auc,0.856968


In [9]:
try:
    import gensim
    from gensim.models import Word2Vec

    print("Gensim version:", gensim.__version__)
    print("Word2Vec is ready.")

except ImportError:
    print("Gensim is not installed.")

Gensim version: 4.4.0
Word2Vec is ready.


In [10]:
def tokenize_code(text):
    text = str(text)

    # Extract identifiers, Java-like tokens, and numbers.
    tokens = re.findall(
        r'[A-Za-z_$][A-Za-z0-9_$]*|[0-9]+|==|!=|<=|>=|&&|\|\||[^\s]',
        text
    )

    return [token.lower() for token in tokens if len(token) > 1]


tokenized_diffs = X_text.apply(tokenize_code)

print("Tokenization complete.")
print("Example tokens:")
print(tokenized_diffs.iloc[0][:40])

Tokenization complete.
Example tokens:
['diff', 'git', 'src', 'test', 'java', 'net', 'spy', 'memcached', 'membaseclienttest', 'java', 'src', 'test', 'java', 'net', 'spy', 'memcached', 'membaseclienttest', 'java', 'index', '39', 'eb8296efb79e26c110f904ddc768edd2ee6416', 'f1366caab3ee394d142b93446952c524365febd8', '100644', 'src', 'test', 'java', 'net', 'spy', 'memcached', 'membaseclienttest', 'java', 'src', 'test', 'java', 'net', 'spy', 'memcached', 'membaseclienttest', 'java', '62']


In [11]:
from gensim.models import Word2Vec

def create_word2vec_features(train_tokens, test_tokens):

    # Train Word2Vec using training-fold data only.
    w2v_model = Word2Vec(
        sentences=train_tokens,
        vector_size=100,
        window=5,
        min_count=2,
        workers=1,
        sg=1,
        epochs=10,
        seed=RANDOM_STATE
    )

    def average_vectors(tokenized_documents):
        vectors = []

        for tokens in tokenized_documents:
            token_vectors = [
                w2v_model.wv[token]
                for token in tokens
                if token in w2v_model.wv
            ]

            if token_vectors:
                vectors.append(np.mean(token_vectors, axis=0))
            else:
                vectors.append(np.zeros(100))

        return np.array(vectors)

    X_train_vectors = average_vectors(train_tokens)
    X_test_vectors = average_vectors(test_tokens)

    return X_train_vectors, X_test_vectors


print("Word2Vec embedding function ready.")

Word2Vec embedding function ready.


In [12]:
def evaluate_word2vec(X_text, y, groups):

    fold_results = []

    oof_predictions = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y), dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X_text, y, groups), start=1
    ):

        train_tokens = tokenized_diffs.iloc[train_idx].tolist()
        test_tokens = tokenized_diffs.iloc[test_idx].tolist()

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        X_train, X_test = create_word2vec_features(
            train_tokens,
            test_tokens
        )

        ros = RandomOverSampler(random_state=RANDOM_STATE)

        X_train_resampled, y_train_resampled = ros.fit_resample(
            X_train,
            y_train
        )

        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

        model.fit(X_train_resampled, y_train_resampled)

        predictions = model.predict(X_test)
        scores = model.predict_proba(X_test)[:, 1]

        oof_predictions[test_idx] = predictions
        oof_scores[test_idx] = scores

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "pr_auc": average_precision_score(y_test, scores)
        })

        print(f"Completed Word2Vec fold {fold}/5")

    fold_df = pd.DataFrame(fold_results)

    mean_metrics = fold_df.drop(columns="fold").mean()

    pooled_metrics = {
        "accuracy": accuracy_score(y, oof_predictions),
        "balanced_accuracy": balanced_accuracy_score(y, oof_predictions),
        "precision": precision_score(y, oof_predictions, zero_division=0),
        "recall": recall_score(y, oof_predictions, zero_division=0),
        "f1": f1_score(y, oof_predictions, zero_division=0),
        "pr_auc": average_precision_score(y, oof_scores)
    }

    print("\nWord2Vec + Logistic Regression + Random Oversampling")
    print("\nMean 5-fold metrics:")
    display(mean_metrics.to_frame("Mean Score"))

    print("\nPooled out-of-fold metrics:")
    display(pd.Series(pooled_metrics, name="Pooled OOF Score").to_frame())

    return {
        "experiment": "Word2Vec + Logistic Regression + Random Oversampling",
        "fold_results": fold_df,
        "mean_metrics": mean_metrics,
        "pooled_metrics": pooled_metrics,
        "oof_predictions": oof_predictions,
        "oof_scores": oof_scores
    }


word2vec_results = evaluate_word2vec(X_text, y, groups)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed Word2Vec fold 1/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed Word2Vec fold 2/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed Word2Vec fold 3/5


... (3 outputs left out)

Completed Word2Vec fold 5/5

Word2Vec + Logistic Regression + Random Oversampling

Mean 5-fold metrics:


,Mean Score
accuracy,0.758769
balanced_accuracy,0.768516
precision,0.472359
recall,0.785946
f1,0.589501
pr_auc,0.612589



Pooled out-of-fold metrics:


,Pooled OOF Score
accuracy,0.758737
balanced_accuracy,0.768400
precision,0.472393
recall,0.785714
f1,0.590038
pr_auc,0.596707


In [13]:
import re

SECURITY_PATTERNS = {
    "cryptography": [
        r"\bCipher\b",
        r"\bMessageDigest\b",
        r"\bSecretKey\b",
        r"\bKeyGenerator\b",
        r"\bMac\b",
        r"\bSignature\b",
        r"\bSecureRandom\b",
        r"\bKeyStore\b"
    ],

    "authentication": [
        r"\bAuthenticator\b",
        r"\bCredentials\b",
        r"\bPrincipal\b",
        r"\bAuthentication\b",
        r"\bLoginContext\b",
        r"\bPasswordAuthentication\b"
    ],

    "authorization": [
        r"\bPermission\b",
        r"\bAccessController\b",
        r"\bAccessControlContext\b",
        r"\bACL\b",
        r"\bRole\b",
        r"\bAuthorization\b"
    ],

    "secure_communication": [
        r"\bSSLContext\b",
        r"\bSSLSocket\b",
        r"\bTrustManager\b",
        r"\bX509TrustManager\b",
        r"\bHostnameVerifier\b",
        r"\bTLS\b",
        r"\bHttpsURLConnection\b"
    ],

    "sql_protection": [
        r"\bPreparedStatement\b",
        r"\bprepareStatement\s*\(",
        r"\bsetString\s*\(",
        r"\bsetInt\s*\(",
        r"\bCallableStatement\b"
    ],

    "input_validation": [
        r"\bvalidate\w*",
        r"\bsanitize\w*",
        r"\bPattern\.compile\b",
        r"\bMatcher\b",
        r"\bcheckNotNull\b",
        r"\brequireNonNull\b",
        r"\bIllegalArgumentException\b"
    ]
}


def extract_added_lines(diff):
    lines = str(diff).splitlines()

    added = [
        line[1:]
        for line in lines
        if line.startswith("+") and not line.startswith("+++")
    ]

    return "\n".join(added)


def extract_security_indicators(diff):
    added_code = extract_added_lines(diff)

    features = {}

    for category, patterns in SECURITY_PATTERNS.items():
        matches = []

        for pattern in patterns:
            matches.extend(
                re.findall(pattern, added_code, flags=re.IGNORECASE)
            )

        features[f"{category}_count"] = len(matches)
        features[f"{category}_present"] = int(len(matches) > 0)

    features["total_security_api_count"] = sum(
        features[f"{category}_count"]
        for category in SECURITY_PATTERNS
    )

    return features


print("Security API indicator extraction functions ready.")

Security API indicator extraction functions ready.


In [14]:
api_features = pd.DataFrame(
    X_text.apply(extract_security_indicators).tolist()
)

print("API feature dataset shape:", api_features.shape)

print("\nFeature columns:")
print(api_features.columns.tolist())

print("\nFeature summary:")
display(api_features.describe().round(3))

print("\nNumber of revisions containing each indicator:")
display(
    api_features[
        [f"{category}_present" for category in SECURITY_PATTERNS]
    ].sum().to_frame("Revisions with indicator")
)

print("\nSecurity-related vs. non-security-related indicator averages:")
display(
    api_features.assign(security_label=y.values)
    .groupby("security_label")
    .mean()
    .round(3)
)

API feature dataset shape: (1774, 13)

Feature columns:
['cryptography_count', 'cryptography_present', 'authentication_count', 'authentication_present', 'authorization_count', 'authorization_present', 'secure_communication_count', 'secure_communication_present', 'sql_protection_count', 'sql_protection_present', 'input_validation_count', 'input_validation_present', 'total_security_api_count']

Feature summary:


,cryptography_count,cryptography_present,authentication_count,authentication_present,authorization_count,authorization_present,secure_communication_count,secure_communication_present,sql_protection_count,sql_protection_present,input_validation_count,input_validation_present,total_security_api_count
count,1774.000,1774.000,1774.000,1774.000,1774.000,1774.000,1774.0,1774.0,1774.000,1774.000,1774.000,1774.000,1774.000
mean,0.156,0.031,0.532,0.046,1.457,0.155,0.0,0.0,0.007,0.003,0.581,0.105,2.732
std,1.302,0.173,5.775,0.210,7.877,0.362,0.0,0.0,0.163,0.058,2.848,0.306,11.829
min,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,0.000
50%,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,0.000
75%,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,1.000
max,27.000,1.000,132.000,1.000,154.000,1.000,0.0,0.0,6.000,1.000,42.000,1.000,185.000



Number of revisions containing each indicator:


,Revisions with indicator
cryptography_present,55
authentication_present,82
authorization_present,275
secure_communication_present,0
sql_protection_present,6
input_validation_present,186



Security-related vs. non-security-related indicator averages:


,cryptography_count,cryptography_present,authentication_count,authentication_present,authorization_count,authorization_present,secure_communication_count,secure_communication_present,sql_protection_count,sql_protection_present,input_validation_count,input_validation_present,total_security_api_count
security_label,,,,,,,,,,,,,
0,0.060,0.012,0.042,0.023,0.288,0.075,0.0,0.0,0.000,0.000,0.28,0.066,0.670
1,0.492,0.097,2.258,0.128,5.579,0.439,0.0,0.0,0.033,0.015,1.64,0.242,10.003


In [15]:
api_results = evaluate_text_model(
    X_text=X_text,
    y=y,
    groups=groups,
    experiment_name="TF-IDF + Security API Indicators + Random Oversampling",
    extra_features=api_features
)

Completed fold 1/5
Completed fold 2/5
Completed fold 3/5
Completed fold 4/5
Completed fold 5/5

TF-IDF + Security API Indicators + Random Oversampling

Mean 5-fold metrics:


,Mean Score
accuracy,0.896834
balanced_accuracy,0.866093
precision,0.745311
recall,0.811068
f1,0.776324
pr_auc,0.845922



Pooled out-of-fold metrics:


,Pooled OOF Score
accuracy,0.896843
balanced_accuracy,0.866177
precision,0.744731
recall,0.811224
f1,0.776557
pr_auc,0.845963


In [16]:
experiment_results = [
    baseline_results,
    word2vec_results,
    api_results
]

comparison_rows = []

for result in experiment_results:
    row = {
        "Experiment": result["experiment"]
    }

    for metric, value in result["mean_metrics"].items():
        row[f"Mean {metric}"] = value

    for metric, value in result["pooled_metrics"].items():
        row[f"Pooled {metric}"] = value

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

display(
    comparison_df.set_index("Experiment").round(4)
)

,Mean accuracy,Mean balanced_accuracy,Mean precision,Mean recall,Mean f1,Mean pr_auc,Pooled accuracy,Pooled balanced_accuracy,Pooled precision,Pooled recall,Pooled f1,Pooled pr_auc
Experiment,,,,,,,,,,,,
Baseline: TF-IDF + Logistic Regression + Random Oversampling,0.8856,0.8662,0.7057,0.8316,0.7623,0.8616,0.8856,0.8663,0.7041,0.8316,0.7626,0.8570
Word2Vec + Logistic Regression + Random Oversampling,0.7588,0.7685,0.4724,0.7859,0.5895,0.6126,0.7587,0.7684,0.4724,0.7857,0.5900,0.5967
TF-IDF + Security API Indicators + Random Oversampling,0.8968,0.8661,0.7453,0.8111,0.7763,0.8459,0.8968,0.8662,0.7447,0.8112,0.7766,0.8460


In [17]:
from scipy.sparse import hstack, csr_matrix

def evaluate_combined_features(X_text, y, groups, api_features):

    fold_results = []

    oof_predictions = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y), dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X_text, y, groups), start=1
    ):

        train_text = X_text.iloc[train_idx].fillna("")
        test_text = X_text.iloc[test_idx].fillna("")

        train_tokens = tokenized_diffs.iloc[train_idx].tolist()
        test_tokens = tokenized_diffs.iloc[test_idx].tolist()

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # 1. TF-IDF features
        vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            stop_words="english"
        )

        X_train_tfidf = vectorizer.fit_transform(train_text)
        X_test_tfidf = vectorizer.transform(test_text)

        # 2. Word2Vec features (training fold only)
        X_train_w2v, X_test_w2v = create_word2vec_features(
            train_tokens,
            test_tokens
        )

        # 3. Security API indicators
        X_train_api = csr_matrix(
            api_features.iloc[train_idx].to_numpy()
        )

        X_test_api = csr_matrix(
            api_features.iloc[test_idx].to_numpy()
        )

        # Combine all feature representations
        X_train_combined = hstack([
            X_train_tfidf,
            csr_matrix(X_train_w2v),
            X_train_api
        ]).tocsr()

        X_test_combined = hstack([
            X_test_tfidf,
            csr_matrix(X_test_w2v),
            X_test_api
        ]).tocsr()

        # Oversample training data only
        ros = RandomOverSampler(random_state=RANDOM_STATE)

        X_train_resampled, y_train_resampled = ros.fit_resample(
            X_train_combined,
            y_train
        )

        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

        model.fit(X_train_resampled, y_train_resampled)

        predictions = model.predict(X_test_combined)
        scores = model.predict_proba(X_test_combined)[:, 1]

        oof_predictions[test_idx] = predictions
        oof_scores[test_idx] = scores

        fold_results.append({
            "fold": fold,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "pr_auc": average_precision_score(y_test, scores)
        })

        print(f"Completed combined experiment fold {fold}/5")

    fold_df = pd.DataFrame(fold_results)

    mean_metrics = fold_df.drop(columns="fold").mean()

    pooled_metrics = {
        "accuracy": accuracy_score(y, oof_predictions),
        "balanced_accuracy": balanced_accuracy_score(y, oof_predictions),
        "precision": precision_score(y, oof_predictions, zero_division=0),
        "recall": recall_score(y, oof_predictions, zero_division=0),
        "f1": f1_score(y, oof_predictions, zero_division=0),
        "pr_auc": average_precision_score(y, oof_scores)
    }

    print("\nCombined TF-IDF + Word2Vec + API Indicators")
    print("\nMean 5-fold metrics:")
    display(mean_metrics.to_frame("Mean Score"))

    print("\nPooled out-of-fold metrics:")
    display(pd.Series(pooled_metrics, name="Pooled OOF Score").to_frame())

    return {
        "experiment": "TF-IDF + Word2Vec + API Indicators",
        "fold_results": fold_df,
        "mean_metrics": mean_metrics,
        "pooled_metrics": pooled_metrics,
        "oof_predictions": oof_predictions,
        "oof_scores": oof_scores
    }


combined_results = evaluate_combined_features(
    X_text,
    y,
    groups,
    api_features
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed combined experiment fold 1/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed combined experiment fold 2/5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Completed combined experiment fold 3/5


... (3 outputs left out)

Completed combined experiment fold 5/5

Combined TF-IDF + Word2Vec + API Indicators

Mean 5-fold metrics:


,Mean Score
accuracy,0.890077
balanced_accuracy,0.867237
precision,0.719501
recall,0.826388
f1,0.767783
pr_auc,0.855079



Pooled out-of-fold metrics:


,Pooled OOF Score
accuracy,0.890079
balanced_accuracy,0.867317
precision,0.718404
recall,0.826531
f1,0.768683
pr_auc,0.853417


In [18]:
all_results = [
    baseline_results,
    word2vec_results,
    api_results,
    combined_results
]

final_comparison = []

for result in all_results:

    row = {"Experiment": result["experiment"]}

    for metric, value in result["mean_metrics"].items():
        row[f"Mean {metric}"] = value

    for metric, value in result["pooled_metrics"].items():
        row[f"Pooled {metric}"] = value

    final_comparison.append(row)

final_comparison_df = pd.DataFrame(final_comparison)

display(
    final_comparison_df.set_index("Experiment").round(4)
)

,Mean accuracy,Mean balanced_accuracy,Mean precision,Mean recall,Mean f1,Mean pr_auc,Pooled accuracy,Pooled balanced_accuracy,Pooled precision,Pooled recall,Pooled f1,Pooled pr_auc
Experiment,,,,,,,,,,,,
Baseline: TF-IDF + Logistic Regression + Random Oversampling,0.8856,0.8662,0.7057,0.8316,0.7623,0.8616,0.8856,0.8663,0.7041,0.8316,0.7626,0.8570
Word2Vec + Logistic Regression + Random Oversampling,0.7588,0.7685,0.4724,0.7859,0.5895,0.6126,0.7587,0.7684,0.4724,0.7857,0.5900,0.5967
TF-IDF + Security API Indicators + Random Oversampling,0.8968,0.8661,0.7453,0.8111,0.7763,0.8459,0.8968,0.8662,0.7447,0.8112,0.7766,0.8460
TF-IDF + Word2Vec + API Indicators,0.8901,0.8672,0.7195,0.8264,0.7678,0.8551,0.8901,0.8673,0.7184,0.8265,0.7687,0.8534


In [19]:
from pathlib import Path

OUTPUT_DIR = Path(
    "results/machine_learning/feature_extension"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_comparison_df.to_csv(
    OUTPUT_DIR / "feature_extension_comparison.csv",
    index=False
)

for result in all_results:

    safe_name = (
        result["experiment"]
        .lower()
        .replace(" + ", "_")
        .replace(" ", "_")
    )

    result["fold_results"].to_csv(
        OUTPUT_DIR / f"{safe_name}_fold_results.csv",
        index=False
    )

print("Experiment results saved to:", OUTPUT_DIR)

Experiment results saved to: results/machine_learning/feature_extension
